In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eigvals
%matplotlib agg   # to prevent images from showing inside the notebook

"""
To be completed!
Field-Golubitsky symmetric attractors
https://paulbourke.net/fractals/symmetryinchaos/
profconradi.com

modified:
     f(z) = z (a_0 + a_1 z z* + a_2 Re[z^n] + a_3 |z| Re[(z/|z|)^(n*p) ] ) + a_4 z*^(n-1)
 original:
     f(z) = z (a_0 + a_1 z z* + a_2 Re[z^n] + a_3 i ] ) + a_4 z*^(n-1)
     f(z) = z (lambda + alpha z z* + beta Re[z^n] +  i omega ] ) + gamma z*^(n-1)
     lambda -> a0
     alpha -> a1
     beta -> a2
     omega -> a3
     gamma -> a4
    z is complex; a0, ..., a4 are real parameters; n,p are positive integers?
    A rough guide to parameters.
        - The value of n determines the rotational symmetry of the image, it should be greater than or equal to 2.
        - a0 is usually between -3 and 3, values between -1 and 1 are generally less interesting.
        - a1 should have the opposite sign as a0, and be in the same range.
        - a2 perturbs the attractor, it can be 0.
        - a3 controls bilateral symmetry, 0 will result in mirror symmetry. Otherwise typical values are between -1 and 1.
        - a4 typically between -1 and 1 and not too close to zero.
    
Drawing method:
    1 iterate the map for about 1000 times, to get the boundaries
    2 iterate for about 10-1000 milion times and count a 2D histogram 
    3 draw the 2D histogram
"""

def f_original(z, a0, a1, a2, a3, a4, n, p):
    zc = z.conjugate()
    za = abs(z)
    o = a0 + a1*z*zc + a2*(z**n).real + a3*1j
    o = o*z + a4*zc**(n-1)
    return o

def f_modified(z, a0, a1, a2, a3, a4, n, p):
    zc = z.conjugate()
    za = abs(z)
    o = a0 + a1*z*zc + a2*(z**n).real + a3*za*((z/za)**(n*p)).real
    o = o*z + a4*zc**(n-1)
    return o

def get_filename(a0, a1, a2, a3, a4, n, p, N, resolution_x):
    return f"plankton_a0{a0:.3f}_a1{a1:.3f}_a2{a2:.3f}_a3{a3:.3f}_a4{a4:.3f}_n{n:d}_p{p:d}_N{N:d}_res{resolution_x:d}"

def get_succession(f, z0, N):
    xs = np.empty(N+1, dtype=np.float32)
    ys = np.empty(N+1, dtype=np.float32)
    z = z0
    xs[0] = z.real
    ys[0] = z.imag
    for i in range(1,N+1):
        z = f(z)
        xs[i] = z.real
        ys[i] = z.imag
    return xs, ys

def get_succession_boundaries(f, z0, N, equal=True):
    xmin, xmax = 99999, -99999
    ymin, ymax = 99999, -99999
    z = z0
    for i in range(1,N+1):
        z = f(z)
        xmin = min(xmin, z.real)
        xmax = max(xmax, z.real)
        ymin = min(ymin, z.imag)
        ymax = max(ymax, z.imag)
    if equal:
        xmin = min(xmin,ymin)
        xmax = max(xmax,ymax)
        ymin = xmin
        ymax = xmax
    extent = (xmin, xmax, ymin, ymax)
    return extent

def get_succession_histogram(extent, pad, resolution,  f, z0, N):
    xmin, xmax, ymin, ymax = extent
    xpad, ypad = pad
    Nx, Ny = resolution
    x = np.linspace(xmin-xpad, xmax+xpad, Nx)
    y = np.linspace(ymin-ypad, ymax+ypad, Ny)
    x0,x1, Lx = x[0],x[-1], x[-1]-x[0]
    y0,y1, Ly = y[0],y[-1], y[-1]-y[0]
    counts = np.zeros((Nx,Ny), dtype=np.uint32)
    z = z0
    for i in range(1,N+1):
        if i%10000 == 0:
            print(f"\r [ {i/N*100:.1f}% ({i:d} / {N:d}) ... ]", end='', flush=True)
        z = f(z)
        if z.real>x0 and z.real<x1 and z.imag>y0 and z.imag<y1:
            ix = int((z.real - x0)/Lx * Nx)
            iy = int((z.imag - y0)/Ly * Ny)
            counts[ix,iy] += 1
    print(" completed", flush=True)
    return counts

def plot_histogram(counts, cmap="gnuplot", log=True):
    fig,ax = plt.subplots(figsize=(16,16), dpi=300)
    img = counts
    if log:
        sel=(counts>0)
        img[sel] = np.log(img[sel])
    ax.imshow(img, aspect="equal", cmap=cmap) #extent=extent)
    ax.set(xticks=[], yticks=[])
    fig.tight_layout()
    return fig,ax

"""
A rough guide to parameters.
        - The value of n determines the rotational symmetry of the image, it should be greater than or equal to 2.
        - a0 is usually between -3 and 3, values between -1 and 1 are generally less interesting.
        - a1 should have the opposite sign as a0, and be in the same range.
        - a2 perturbs the attractor, it can be 0.
        - a3 controls bilateral symmetry, 0 will result in mirror symmetry. Otherwise typical values are between -1 and 1.
        - a4 typically between -1 and 1 and not too close to zero.
    lambda -> a0
    alpha -> a1
    beta -> a2
    omega -> a3
    gamma -> a4
"""

def main(
    f = f_original,
    a0 = -1.86,
    a1 = 2.0,
    a2 = 0.0,
    a3 = 0.1,
    a4 = 1.0,
    n = 4,
    p = 1, #useless if f == f_original
    z0 = 0.1+0.3214j,
    xpad = 0.0,
    xres = 400,
    N0 = 1000,
    N = 1_000_000,
    cmap = "gnuplot",
):
    pad = (xpad, xpad)
    resolution = (xres, xres)
    params = (a0, a1, a2, a3, a4, n, p)
    func = lambda z: f(z,*params)
    fileroot = f"{get_filename(*params, N, xres)}"
    print(fileroot + " ... ", flush=True)
    
    """
    xs, ys = get_succession(func, z0, N0)
    z0 = xs[-1] + ys[-1]*1j
    xs, ys = get_succession(func, z0, N)
    fig,ax = plt.subplots()
    ax.axvline(0, color='k')
    ax.axhline(0, color='k')
    ax.scatter(xs,ys, ec='none', c=np.arange(len(xs)), marker='.', s=4, alpha=0.2)
    ax.set(facecolor='white')
    plt.show()
    """
    
    try:
        counts = np.loadtxt(f"{fileroot}.dat")
        print(" loaded", flush=True)
    except FileNotFoundError:
        extent = get_succession_boundaries(func, z0, N0)
        print(" extent is: %f %f %f %f"%extent)
        print(" computing ... ", flush=True)
        counts = get_succession_histogram(extent, pad, resolution, func, z0, N)
        np.savetxt(f"{fileroot}.dat", counts, fmt="%u")
    print(" plotting ...")
    fig,ax = plot_histogram(counts, cmap=cmap)
    for fmt in ["png"]:
        fig.savefig(f"{fileroot}.{fmt}")
    print(" saved image(s)")

In [24]:
%%time
args = dict(
    f = f_original,
    a0 = -1.86,
    a1 = 2.0,
    a2 = 0.0,
    a3 = 0.1,
    a4 = 1.0,
    n = 4,
    p = 1, #useless if f == f_original
    xres = 800,
    N = 30_000_000,
)
main(**args)

plankton_a0-1.860_a12.000_a20.000_a30.100_a41.000_n4_p1_N30000000_res800 ... 
 extent is: -0.897268 0.896075 -0.888217 0.894896
 computing ... 
 [ 100.0% (30000000 / 30000000) ... ] completed
 plotting ...
 saved image(s)
CPU times: total: 7min 1s
Wall time: 7min 2s


In [20]:
%%time
args = dict(
    f = f_original,
    a0 = 1.52,
    a1 = -1.21,
    a2 = -0.091,
    a3 = 0.0,
    a4 = -0.8005,
    n = 3,
    p = 1, #useless if f == f_original
    xres = 800,
    N = 500_000,
)
main(**args)

plankton_a01.520_a1-1.210_a2-0.091_a30.000_a4-0.800_n3_p1_N500000_res800 ... 
 extent is: -1.203812 0.941473 -1.085918 1.086663
 computing ... 
 [ 100.0% (500000 / 500000) ... ] completed
 plotting ...
 saved image(s)
CPU times: total: 14.4 s
Wall time: 14.7 s


In [18]:
%%time
for xres in [800,1200]:
    args = dict(
        f = f_original,
        a0 = 1.415,
        a1 = -1.21,
        a2 = -0.001,
        a3 = 0.0,
        a4 = -0.8005,
        n = 5,
        p = 1, #useless if f == f_original
        xres = xres,
        N = 500_000,
    )
    main(**args)

plankton_a01.415_a1-1.210_a2-0.001_a30.000_a4-0.800_n5_p1_N500000_res800 ... 
 extent is: -0.969089 0.879308 -0.946595 0.951597
 computing ... 
 [ 100.0% (500000 / 500000) ... ] completed
 plotting ...
 saved image(s)
plankton_a01.415_a1-1.210_a2-0.001_a30.000_a4-0.800_n5_p1_N500000_res1200 ... 
 extent is: -0.969089 0.879308 -0.946595 0.951597
 computing ... 
 [ 100.0% (500000 / 500000) ... ] completed
 plotting ...
 saved image(s)
CPU times: total: 31.6 s
Wall time: 32 s
